# ProtRL toy workflow: SFT + GRPO on protein sequences

This notebook is a compact, didactical version of the working workflow.

It does three things:

1. creates a toy mutation dataset from one protein sequence;
2. does a simple supervised fine-tuning step on the positive examples;
3. runs ProtRL GRPO using both positive and negative rewards.

The dataset and rewards here are artificial. Replace the random reward with a real biological score when moving beyond the toy example.


## 1. Install and clone dependencies


In [ ]:
!pip uninstall -y torchao
!pip install -U transformers datasets peft accelerate trl torch --quiet
![ -d /content/ProtRL ] || git clone https://github.com/AI4PDLab/ProtRL.git --branch restructuring


## 2. Imports and basic configuration

In [ ]:
import os
import sys
import random
import csv

import torch
import pandas as pd
from datasets import Dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

from peft import LoraConfig, get_peft_model

sys.path.insert(0, "/content/ProtRL")
from src.ProtRL_Trainer import ProtRLTrainingArgument
from src.pLM_GRPO import ProtRL_GRPOTrainer


SEED = 42
MODEL_NAME = "AI4PD/ProtGPT3-112M"
PROMPT = "M"

WT_SEQUENCE = "HGEGTFTSDLSKQMEEEAVRLFIEWLKNGGPSSGAPPPS"
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"

N_MUTANTS = 1000
MUTATION_RATE = 0.10
DATASET_CSV = "dataset.csv"

random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("Wild-type sequence length:", len(WT_SEQUENCE))


## 3. Load ProtGPT3

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")
print("BOS:", tokenizer.bos_token, "EOS:", tokenizer.eos_token, "PAD:", tokenizer.pad_token)


## 4. Create a toy mutation dataset

Each sequence is a random mutant of the starting sequence.

The reward is randomly assigned as `+1` or `-1` only for demonstration. In a real experiment, this should come from your scoring function, classifier, assay, or oracle.


In [ ]:
rows = []

for _ in range(N_MUTANTS):
    mutant = list(WT_SEQUENCE)

    for i, aa in enumerate(mutant):
        if random.random() < MUTATION_RATE:
            mutant[i] = random.choice(AMINO_ACIDS.replace(aa, ""))

    sequence = "".join(mutant)
    reward = random.choice([1, -1])
    rows.append([sequence, reward])

with open(DATASET_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["sequence", "reward"])
    writer.writerows(rows)

df = pd.read_csv(DATASET_CSV)
df.head()


## 5. Supervised fine-tuning on positive examples

Here we apply Supervise Finetuning with LoRA for efficent computing. We train only on good examples. 


In [ ]:
positive_df = df[df["reward"] == 1].reset_index(drop=True)
positive_df["text"] = PROMPT + positive_df["sequence"].astype(str)

sft_dataset = Dataset.from_pandas(positive_df[["text"]], preserve_index=False)

def tokenize_sft(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=len(WT_SEQUENCE) + 8,
    )

tokenized_sft = sft_dataset.map(
    tokenize_sft,
    batched=True,
    remove_columns=["text"],
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print("Positive SFT examples:", len(tokenized_sft))


## 6. Add LoRA adapters

LoRA trains only a small number of adapter parameters instead of updating the full model. This will allow you to fit training in a smaller GPU


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 7. Run the SFT step

This gives the model an initial push toward sequences labeled as positive in the toy dataset.


In [ ]:
sft_args = TrainingArguments(
    output_dir="./sft_results",
    report_to="none",
    logging_steps=10,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_strategy="no",
)

sft_trainer = Trainer(
    model=model,
    args=sft_args,
    train_dataset=tokenized_sft,
    data_collator=data_collator,
)

sft_trainer.train()

model.save_pretrained("./lora_positive_reward_model")
tokenizer.save_pretrained("./lora_positive_reward_model")


## 8. Generate sequences from the SFT model

In [ ]:
def generate_sequences(n=5):
    inputs = tokenizer(PROMPT, return_tensors="pt").to(device)

    with torch.no_grad():
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=len(WT_SEQUENCE),
            do_sample=True,
            num_return_sequences=n,
            top_p=0.95,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.batch_decode(gen_ids, skip_special_tokens=True)


generated = generate_sequences(n=5)

for i, seq in enumerate(generated):
    print(f">{i}")
    print(seq.replace(" ", ""))
    print()


## 9. Prepare the GRPO dataset

In [ ]:
rl_df = df.copy()
rl_df["prompt"] = PROMPT
rl_df["completion"] = rl_df["sequence"].astype(str)
rl_df["reward"] = rl_df["reward"].astype(float)

rl_dataset = Dataset.from_pandas(
    rl_df[["prompt", "completion", "reward"]],
    preserve_index=False,
)

split = rl_dataset.train_test_split(test_size=0.2, seed=SEED, shuffle=True)

train_dataset = split["train"]
eval_dataset = split["test"]

print("Train examples:", len(train_dataset))
print("Eval examples:", len(eval_dataset))
print(train_dataset[0])


## 10. Run ProtRL GRPO

In [ ]:
grpo_args = ProtRLTrainingArgument(
    output_dir="RL",
    report_to="none",
    logging_steps=10,
    num_train_epochs=1,
    dataloader_num_workers=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
)

grpo_trainer = ProtRL_GRPOTrainer(
    model=model,
    args=grpo_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

grpo_trainer.train()
grpo_trainer.save_model("RL/final_model")

torch.cuda.empty_cache()
